# ✋ Hand AI – ASL A-Z Training (26 classes)
Train a MobileNetV2-based Keras model on the ASL Alphabet dataset.
Upload your `asl_alphabet_train.zip` to Colab or mount Google Drive.

**Output**: `hand_model.keras` with 26 output classes (A–Z)
**Classes**: A, B, C, D, E, F, G, H, I, J, K, L, M, N, O, P, Q, R, S, T, U, V, W, X, Y, Z

In [ ]:
# ── Step 1: Mount Google Drive (optional) ──────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')

# ── Step 2: Upload dataset zip ─────────────────────────────────────────────
# from google.colab import files
# uploaded = files.upload()   # Upload asl_alphabet_train.zip

# ── OR: download from Kaggle directly ─────────────────────────────────────
# !pip install kaggle -q
# Upload your kaggle.json first, then:
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d grassknoted/asl-alphabet -q
# !unzip -q asl-alphabet.zip -d /content/asl_data

print('Setup complete. Set TRAIN_DIR below to your dataset path.')

In [ ]:
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print('TensorFlow version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
# IMPORTANT: Set this to your dataset folder (the folder containing A/, B/, ..., Z/ subfolders)
TRAIN_DIR = '/content/asl_data/asl_alphabet_train/asl_alphabet_train'  # adjust if needed

IMG_SIZE  = 224
BATCH     = 32
EPOCHS    = 10
NUM_CLASSES = 26  # A-Z only

# Only keep A-Z (26 classes), skip 'del', 'nothing', 'space'
CLASSES = list('ABCDEFGHIJKLMNOPQRSTUVWXYZ')

print(f'Classes ({len(CLASSES)}):', CLASSES)

In [ ]:
# ── Data Generators ────────────────────────────────────────────────────────
datagen = ImageDataGenerator(
    rescale=1./127.5,
    preprocessing_function=lambda x: x - 1.0,  # → [-1, 1]
    validation_split=0.1,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=False,  # Don't flip - ASL signs are hand-specific
)

train_gen = datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode='categorical',
    subset='training',
    classes=CLASSES,  # Only load A-Z
)

val_gen = datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode='categorical',
    subset='validation',
    classes=CLASSES,
)

print('Class index map:', train_gen.class_indices)
print('Training samples:', train_gen.samples)
print('Validation samples:', val_gen.samples)

In [ ]:
# ── Save classes.json (CRITICAL: must match model output order) ────────────
# class_indices = {'A': 0, 'B': 1, ...} → invert to [class_at_0, class_at_1, ...]
idx_to_class = {v: k for k, v in train_gen.class_indices.items()}
classes_list = [idx_to_class[i] for i in range(len(idx_to_class))]

with open('classes.json', 'w') as f:
    json.dump(classes_list, f)

print('classes.json saved:', classes_list)

In [ ]:
# ── Build Model ────────────────────────────────────────────────────────────
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False  # Freeze base initially

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax'),  # ← 26 classes
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

In [ ]:
# ── Phase 1: Train head only ───────────────────────────────────────────────
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5),
    ]
)

In [ ]:
# ── Phase 2: Fine-tune (optional, for better accuracy) ────────────────────
base_model.trainable = True
# Freeze bottom layers, only fine-tune the top
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # Lower LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=5,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    ]
)

In [ ]:
# ── Save Model ────────────────────────────────────────────────────────────
model.save('hand_model.keras')
print('✅ Model saved as hand_model.keras')
print(f'   Output classes: {NUM_CLASSES}')

# Download both files
from google.colab import files
files.download('hand_model.keras')
files.download('classes.json')

## After downloading:
1. Place `hand_model.keras` → `ai_models/hand-ai/saved_models/hand_model.keras` (replace existing)
2. Place `classes.json` → `ai_models/hand-ai/saved_models/classes.json` (replace existing)
3. Restart the hand-ai server — it will auto-load the new 26-class model!